## How to run minibinder + scFv design fully end-to-end, locally.

This is a local-GPU adaptation of Biohub's [original tutorial](https://github.com/biohub/esm/tree/main/cookbook/tutorials), which uses Modal to parallelize design jobs on cloud GPUs. This version runs `binder_design.py`'s design loop directly on this machine's GPU instead — no Modal account or spend required, at the cost of running jobs sequentially instead of in parallel.

Biohub used this approach to design minibinders and scFvs against five therapeutically relevant targets — PDGFRB, EGFR, PD-L1, CD45, and CTLA4 — spanning receptor tyrosine kinases, immune checkpoints, and cell-surface phosphatases. Binders exhibit nanomolar affinity, target specificity, and functional activity in laboratory assays, per the ESMC and ESMFold2 paper titled ["Language Modeling Materializes a World Model of Protein Biology"](https://biohub.ai/papers/esm_protein.pdf).

> **Model size tradeoff:** Biohub's flagship checkpoints (`ESMFold2-Experimental-Fast`/`ESMFold2-Experimental`) are paired with the 6B-parameter ESMC language model — a ~24GB download/VRAM commitment for the shared LM alone. To avoid that, `binder_design.py` in this folder is configured to use Biohub's smaller scaling-study checkpoints instead (`ESMFold2-Experimental-Fast-base300M-step1500k` for the inversion model, `-step1000k`/`-step1500k` as critics), paired with the 300M ESMC. This is ~20x smaller, but per Biohub's own scaling-law results, designs from these smaller/earlier checkpoints will likely be lower quality (worse success rate, less accurate structures) than the flagship. If quality matters more than local resource use, switch back to the flagship names and accept the 6B download (or use Modal instead).

**You'll need:** a target protein sequence (or pick one of the built-in presets), and a local GPU.

**You'll get:** a file of top-ranked designed binder sequences, plus 3D structures of the predicted complexes that you can view in the notebook.

**Workflow:**
1. **Setup** (one-time): install dependencies, load the model ensemble onto the GPU
2. **Try one job**: pick a target and binder type, run a single design end-to-end as a sanity check
3. **Run a sweep**: run many seeds sequentially to produce real candidates
4. **Pick the designs to order**: filter and rank, save the shortlist


### 1. Setup

- note that waluigi:8890 is for esm2 (other 2 r for boltzgen)

In [2]:
# Dependencies are installed into the venv by start_jupyter_esmfold2.bash
# (via `uv pip install`), not here. Bare `! pip install` in this kernel can
# resolve to the system pip instead of the venv's (uv-created venvs don't
# ship a `pip` binary), which silently installs into the wrong place. If you
# ever do need to install something from a cell, use sys.executable:
import sys
print(sys.executable)
# !{sys.executable} -m pip install <package>

/home/bridget/esmfold2_venv/bin/python


In [3]:
%cd ~/notebooks

/home/bridget/notebooks


### Imports

In [4]:
from itertools import product
from pathlib import Path

import pandas as pd
import py3Dmol
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

from binder_design import ESMFold2Design

ESMC: transformer_engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically — measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack this reaches ~O(100) max-diff on the unnormalized residual stream. Install xformers (preferred) with `pip install xformers` for a fused attention kernel.
ESMC: flash-attn rotary kernel not installed; falling back to pure-PyTorch RoPE. For faster GPU inference run `pip install flash-attn`.


🚨 No checkpoint found for ESMCForSequenceClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring
🚨 No checkpoint found for ESMCForTokenClassification.forward. Please add a `checkpoint` arg to `auto_docstring` or add one in ESMCConfig's docstring


In [5]:
import importlib
import binder_design
importlib.reload(binder_design)
from binder_design import ESMFold2Design

### App setup

`ESMFold2Design()` loads the full model ensemble (inversion models, critics, ESMC) directly onto this machine's GPU. This takes a few minutes and a lot of VRAM — do it once and reuse the `app` object for every design call below, rather than re-instantiating it.

`use_scaling_critics=False` is the default. Setting it to `True` adds extra critic models from the paper that improve selection at the cost of more compute and VRAM per job — likely not viable locally unless you have a lot of headroom.

In [9]:
app = ESMFold2Design()
# Set 'use_scaling_critics' to evaluate with the additional critics.
app.load(use_scaling_critics=False)

`torch_dtype` is deprecated! Use `dtype` instead!


## 2. Try one design job

Run a single job end-to-end before launching a sweep. This is a sanity check that everything is wired up and that the target/scaffold combo you've chosen produces a sensible complex.

Pick **one** of the two options below and run only that cell. (They both define `best_sequences`/`trajectory`/`critic_results`, so running both back-to-back overwrites the first.)

**Option 1** uses a built-in target and binder scaffold. Available targets: `ctla4`, `egfr`, `pdgfrb`, `pd-l1`, `cd45`. Available binder types: `minibinder`, `trastuzumab_framework_vhvl` (an antibody scaffold). Easiest if your target is one of the built-ins.

**Option 2** takes your own target sequence and binder scaffold. In the binder scaffold, `#` means "design this position" and any amino acid letter means "keep this position fixed." For example:
- `"#" * 60` designs a fully free 60-residue minibinder
- A trastuzumab-style antibody scaffold (shown below) fixes the framework regions and lets the model design the CDR loops

If you're designing an antibody, pass `is_antibody=True` so the selection step later uses the antibody-appropriate scoring.

This runs synchronously on your local GPU and blocks until finished — you'll see per-step loss logging printed directly below the cell.

In [ ]:
# ---- Option 1: Use presets. ----
# Relies on the registry in binder_design.py::{TARGET_SEQUENCES,BINDER_PROMPT_FACTORIES}, which can be modified.
# best_sequences, trajectory, critic_results = app.design(target_name="ctla4", binder_name="minibinder")

In [10]:
# ---- Option 2: Provide your own sequences. ----
# Our s100b sequence crop.
s100b_sequence = "SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE"

best_sequences, trajectory, critic_results = app.design(
    target_sequence=s100b_sequence,
    binder_sequence="#" * 50,   # 50 fully-designed residues
)
print("Best sequences:", best_sequences)

2026-06-26 23:34:49,192 INFO: Enabling RDKit 2026.03.3 jupyter extensions


Loading CCD dictionary from /home/bridget/.cache/huggingface/hub/models--biohub--ESMFold2/snapshots/1ebf0e3481a5184eb6171d40615c79e384b48796/ccd.pkl


2026-06-26 23:34:55,954 INFO:   step   0  |  intra_contact_loss=4.5801  inter_contact_loss=3.7846  glob_loss=4.5647  total_loss=8.2359  plm_loss=3.1406  T=0.9999
2026-06-26 23:34:59,221 INFO:   step   5  |  intra_contact_loss=3.8247  inter_contact_loss=3.4136  glob_loss=3.2798  total_loss=7.4782  plm_loss=3.2031  T=0.9961
2026-06-26 23:35:02,353 INFO:   step  10  |  intra_contact_loss=3.5552  inter_contact_loss=3.4194  glob_loss=2.5441  total_loss=7.4961  plm_loss=3.5000  T=0.9869
2026-06-26 23:35:05,669 INFO:   step  15  |  intra_contact_loss=3.5194  inter_contact_loss=3.2648  glob_loss=2.5948  total_loss=7.4267  plm_loss=3.5156  T=0.9725
2026-06-26 23:35:08,803 INFO:   step  20  |  intra_contact_loss=3.4714  inter_contact_loss=3.3161  glob_loss=2.3497  total_loss=7.4887  plm_loss=3.6250  T=0.9529
2026-06-26 23:35:11,955 INFO:   step  25  |  intra_contact_loss=3.5414  inter_contact_loss=3.3171  glob_loss=2.4535  total_loss=7.5137  plm_loss=3.5938  T=0.9284
2026-06-26 23:35:15,108 INFO

Best sequences: ['SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE|LLLLLRGGGAVAALLAALVVLLVPALGGGGSLAAVAAAAVKVVVWLVQLL']


In [11]:
# ---- Inspect result ----
df = pd.DataFrame(critic_results)
df.drop(columns=["logits", "complex"])

,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy
0,False,ESMFold2-Experimental-Fast-base300M-step1000k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.523508,None,0.336345,NaN
1,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.523508,None,0.356215,NaN


In [12]:
# ---- Visualize ----
protein_complex = (
    df[df.critic_name.eq("ESMFold2-Experimental-Fast-base300M-step1500k")].iloc[0].complex
)
(
    py3Dmol.view(width=600, height=600)
    .addModel(protein_complex.to_pdb_string(), "pdb")
    .setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})  # pyright: ignore
    .setStyle({"chain": "B"}, {"cartoon": {"color": "cyan"}})  # pyright: ignore
    .addStyle(  # pyright: ignore
        {"not": {"atom": ["N", "C", "O"]}},
        {"stick": {"colorscheme": "default", "radius": 0.2}},
    )
    .center()  # pyright: ignore
    .zoomTo()  # pyright: ignore
)

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [20]:
# ---- Visualize top 3 designs ----
TOP_SEEDS = [27, 3, 2]
CRITIC = "ESMFold2-Experimental-Fast-base300M-step1500k"

for seed in TOP_SEEDS:
    row = df_result[(df_result.seed == seed) & (df_result.critic_name == CRITIC)].iloc[0]
    scores = df_select.xs(seed, level=1) if seed in df_select.index.get_level_values(1) else None
    binder_seq = row.designed_sequence.split("|")[1]
    proxy = row.distogram_iptm_proxy
    print(f"--- Seed {seed} | iptm_proxy={proxy:.3f} | binder: {binder_seq}")
    if scores is not None and not scores.empty:
        s = scores.iloc[0]
        print(f"    site1={s.contact_site1_helix:.2f}  site2={s.contact_site2_loop:.2f}  site3={s.contact_site3_hydrophobic:.2f}  avoid={s.contact_avoid:.2f}")
    display(
        py3Dmol.view(width=500, height=400)
        .addModel(row.complex.to_pdb_string(), "pdb")
        .setStyle({"chain": "A"}, {"cartoon": {"color": "green"}})
        .setStyle({"chain": "B"}, {"cartoon": {"color": "cyan"}})
        .addStyle({"not": {"atom": ["N", "C", "O"]}}, {"stick": {"colorscheme": "default", "radius": 0.2}})
        .center()
        .zoomTo()
    )


--- Seed 27 | iptm_proxy=0.412 | binder: SLLLLLLKLARKLVQSGNLALAVLVALLAVLLLGGNPQLAVAVVAAVLLQ


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

--- Seed 3 | iptm_proxy=0.352 | binder: LLLLLLLVLVTAVHTALTALTLLGNGGGLLLALLLLLLLLALAVVLLLLV


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

--- Seed 2 | iptm_proxy=0.479 | binder: SSLLKMLLLLLLLLLVLKYLYQNGLLSPSQYQRYKSFLLGMLKMLLRLLS


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Run a sweep for real designs
For real candidates worth ordering, sweep across many seeds (and optionally multiple targets, binder types, or lengths) and select the best.

Edit `line_sweeps` below to define your campaign. Each key is a sweep axis; the notebook runs one job per combination of values. Unlike the Modal version, this runs **sequentially on your local GPU** — there's no cloud parallelism, so wall-clock time is roughly (per-job time from section 2) × (number of jobs). Start with a small number of seeds to estimate timing before committing to a big sweep.

**Before you click Run on the Launch cell, check the printed shape of the dataframe and confirm it's the number of jobs you intended.**

In [13]:
# ---- Config ----
save_dir = Path("sweep")
save_dir.mkdir(exist_ok=True)

s100b_sequence = "SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEEIKEQEVVDKVMETLDNDGDGECDFQEFMAFVAMVTTACHEFFEHE"

line_sweeps = dict(
    target_name=[None],
    target_sequence=[s100b_sequence],
    binder_name=[None],
    binder_sequence=["#" * 50],   # 50 fully-designed residues
    use_scaling_critics=[False],  # must match app.load() above
    seed=list(range(50)),          # 50 seeds → 50 designs (~3 hrs on single GPU)
    batch_size=[1],
)
df = pd.DataFrame(product(*line_sweeps.values()), columns=list(line_sweeps.keys()))
display(df.head(2))
df.shape

,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size
0,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,##############################################...,False,0,1
1,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,##############################################...,False,1,1


(50, 7)

In [14]:
# ---- Run sweep (sequential — single local GPU, no cloud parallelism) ----
result_dfs = []
failures = []
for row in tqdm(list(df.itertuples()), desc="sweep"):
    try:
        for m in app.hf_critic_models.values():
            m.cuda()
        best_sequences, trajectory, critic_results = app.design(
            target_name=row.target_name,
            target_sequence=row.target_sequence,
            binder_name=row.binder_name,
            binder_sequence=row.binder_sequence,
            seed=row.seed,
            batch_size=row.batch_size,
        )
    except Exception as e:
        failures.append((row.Index, str(e)))
        continue
    result_df = pd.DataFrame(critic_results)
    for col in (
        "target_name", "target_sequence", "binder_name", "binder_sequence",
        "use_scaling_critics", "seed", "batch_size",
    ):
        result_df[col] = getattr(row, col)
    result_dfs.append(result_df)

df_result = pd.concat(result_dfs, ignore_index=True)
df_result.drop(columns=["complex", "logits"], errors="ignore").to_parquet(
    save_dir / "manifest.parquet", index=False
)
print(f"Completed {len(df) - len(failures)}/{len(df)} jobs -> {len(df_result)} result rows")
if failures:
    print(f"{len(failures)} jobs failed, e.g.: {failures[0]}")

sweep:   0%|          | 0/50 [00:00<?, ?it/s]

2026-06-26 23:38:11,810 INFO:   step   0  |  intra_contact_loss=4.5812  inter_contact_loss=3.7937  glob_loss=4.5617  total_loss=8.2404  plm_loss=3.1406  T=0.9999
2026-06-26 23:38:14,943 INFO:   step   5  |  intra_contact_loss=3.6819  inter_contact_loss=3.4709  glob_loss=2.9694  total_loss=7.5297  plm_loss=3.3594  T=0.9961
2026-06-26 23:38:18,277 INFO:   step  10  |  intra_contact_loss=3.4249  inter_contact_loss=3.4562  glob_loss=2.3262  total_loss=7.2808  plm_loss=3.3750  T=0.9869
2026-06-26 23:38:21,417 INFO:   step  15  |  intra_contact_loss=3.4001  inter_contact_loss=3.3118  glob_loss=2.3847  total_loss=7.4110  plm_loss=3.5781  T=0.9725
2026-06-26 23:38:24,560 INFO:   step  20  |  intra_contact_loss=3.4901  inter_contact_loss=3.4727  glob_loss=2.4055  total_loss=7.4312  plm_loss=3.4688  T=0.9529
2026-06-26 23:38:27,923 INFO:   step  25  |  intra_contact_loss=3.4835  inter_contact_loss=3.4956  glob_loss=2.4083  total_loss=7.2993  plm_loss=3.3281  T=0.9284
2026-06-26 23:38:31,077 INFO

Completed 50/50 jobs -> 100 result rows


In [15]:
print(len(failures))
print(failures[:3])

0
[]


## 4. Pick the designs to order

This is your final shortlist. The cell below:

1. Combines results from all successful jobs into one dataframe.
2. Filters minibinders to isoelectric point under 6 (helps with solubility and expression). Antibodies pass through unfiltered.
3. Scores each unique designed sequence by averaging `iptm` (interface predicted TM-score, higher is better) and an `iptm_proxy` term across its trajectories.
4. Returns the top 84 designs per (target, binder type), saved to `selection.parquet` inside your `save_dir`.

84 is a plate-friendly number for ordering and screening. The cutoff and the isoelectric-point filter are currently hardcoded inside the cell, so to change them, edit the values directly in the function.


In [17]:
# ---- Select (instrumented) ----

# Tunables ----------------------------------------------------------
PI_THRESHOLD = None      # set e.g. 8.0 to filter by isoelectric point
TOP_N = 84               # designs to keep per (target, binder) group
SCALING_CHECKPOINT_SUBSTRING = "ESMFold2-Experimental-Fast-base"
# -------------------------------------------------------------------

print(f"[1] rows collected from successful jobs: {len(df_result)}")
print(f"    unique designed_sequence: {df_result.designed_sequence.nunique()}")

if df_result.empty:
    raise ValueError(
        "df_result is empty — no successful job results were collected. "
        "Check the failures list printed by the sweep cell upstream."
    )

df_result["binder_sequence"] = df_result.designed_sequence.str.split(r"\|").str[1]
df_result["isoelectric_point"] = [
    ProteinAnalysis(seq).isoelectric_point()
    for seq in tqdm(df_result.binder_sequence.values)
]
print(
    f"[2] pI distribution: min={df_result.isoelectric_point.min():.2f} "
    f"median={df_result.isoelectric_point.median():.2f} "
    f"max={df_result.isoelectric_point.max():.2f}"
)

if PI_THRESHOLD is None:
    df_filter = df_result.copy()
    print("[3] pI filter DISABLED — keeping all designs")
else:
    df_filter = df_result[
        df_result.is_antibody | df_result.isoelectric_point.lt(PI_THRESHOLD)
    ]
    n_drop = df_result.designed_sequence.nunique() - df_filter.designed_sequence.nunique()
    print(
        f"[3] after pI<{PI_THRESHOLD} filter: {df_filter.designed_sequence.nunique()} "
        f"unique designs kept ({n_drop} dropped)"
    )

if df_filter.empty:
    raise ValueError(
        f"All designs were removed by the pI filter. Raise PI_THRESHOLD or set to None."
    )


def select(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    is_scaling = df.critic_name.str.contains(
        SCALING_CHECKPOINT_SUBSTRING, regex=False, na=False
    )
    iptm_proxy = df.distogram_iptm_proxy.where(
        ~df.is_antibody, df.cdr_distogram_iptm_proxy
    )
    df["iptm_score"] = df.iptm.where(~is_scaling)
    df["iptm_proxy_score"] = iptm_proxy.where(is_scaling)

    contact_cols = [c for c in df.columns if c.startswith("contact_site")]
    has_avoid = "contact_avoid" in df.columns
    agg_spec = dict(
        iptm_score=("iptm_score", "mean"),
        iptm_proxy_score=("iptm_proxy_score", "mean"),
        **{c: (c, "mean") for c in contact_cols},
        **(dict(contact_avoid=("contact_avoid", "mean")) if has_avoid else {}),
    )
    scores = df.groupby("designed_sequence", as_index=False).agg(**agg_spec)

    site_contact = scores[contact_cols].fillna(0).mean(axis=1) if contact_cols else 0.0
    avoid_penalty = scores["contact_avoid"].fillna(0) if has_avoid else 0.0
    scores["selection_score"] = (
        0.5 * scores.iptm_proxy_score.fillna(0)
        + 0.3 * site_contact
        - 0.2 * avoid_penalty
    )
    return scores.nlargest(min(len(scores), TOP_N), "selection_score")


df_select = df_filter.groupby(["target_name", "binder_name"], dropna=False).apply(
    select, include_groups=False
)
df_select.to_parquet(save_dir / "selection.parquet", index=False)
print(f"[4] final selection: {len(df_select)} designs -> {save_dir / 'selection.parquet'}")
df_select

[1] rows collected from successful jobs: 100
    unique designed_sequence: 50


  0%|          | 0/100 [00:00<?, ?it/s]

[2] pI distribution: min=4.05 median=10.30 max=12.00
[3] pI filter DISABLED — keeping all designs
[4] final selection: 50 designs -> sweep/selection.parquet


designed_sequence  \
target_name binder_name                                                         
NaN         NaN         27  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        3   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        2   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        5   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        21  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        45  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        25  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        37  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        39  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        40  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        0   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        49  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        23  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        12  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        9   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        19  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        24  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        22  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        14  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        38  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        35  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        4   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        41  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        6   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        47  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        36  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        11  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        17  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        46  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        28  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        32  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        16  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        48  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        10  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        1   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        20  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        13  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        43  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        29  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        8   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        31  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        15  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        33  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        34  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        26  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        18  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        7   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        30  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
          

In [18]:
df_result[df_result.critic_name.eq("ESMFold2-Experimental-Fast-base300M-step1500k")].drop(
    columns=["complex", "logits"]
)

,is_antibody,critic_name,batch_idx,designed_sequence,final_loss,iptm,distogram_iptm_proxy,cdr_distogram_iptm_proxy,target_name,target_sequence,binder_name,binder_sequence,use_scaling_critics,seed,batch_size,contact_site1_helix,contact_site2_loop,contact_site3_hydrophobic,contact_avoid,isoelectric_point
1,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.133523,None,0.427747,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,KKKLLKVAAVIILVVALLVGALLSSGGGGLAKKVIKLALKLVKKVL...,False,0,1,0.375,0.416667,0.555556,0.000000,11.000128
3,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.101623,None,0.288657,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,VKLTRQQLLQLAVAAIVAAAAAAAAAGNSKLVLLLLLLGLLLVLLL...,False,1,1,0.375,0.166667,0.555556,0.217391,11.166198
5,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.746590,None,0.479409,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,SSLLKMLLLLLLLLLVLKYLYQNGLLSPSQYQRYKSFLLGMLKMLL...,False,2,1,0.250,0.333333,0.666667,0.043478,10.214516
7,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.610528,None,0.352241,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,LLLLLLLVLVTAVHTALTALTLLGNGGGLLLALLLLLLLLALAVVL...,False,3,1,0.375,0.250000,0.666667,0.000000,6.741127
9,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.616396,None,0.513187,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,VNAYQIARLVLTAALLGLLLLVLPDKGGSLAEKLLRLLVSVLAELY...,False,4,1,0.250,0.250000,0.555556,0.043478,9.818293
11,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.472350,None,0.376085,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,LLGLLLRVLRKATTVASSLASSSSPSSSTLLLALLLLLVALLALLL...,False,5,1,0.250,0.416667,0.666667,0.173913,10.835024
13,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.510911,None,0.272761,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MNLLLLLLNVILNIANLLLSLYKSGKLSKEQVLVILLLLLLLLLLL...,False,6,1,0.375,0.250000,0.444444,0.086957,9.824482
15,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.492664,None,0.324423,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,LLLLLLLLLLLVLLLSKLVKKLSSSGKNPGAVLALLKTAISMLALL...,False,7,1,0.250,0.250000,0.666667,0.391304,10.602487
17,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.423659,None,0.304553,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,MIWLVLVAAAAAAAAASGNPGLAVAVVTAAAYLAANGGGLAALIVA...,False,8,1,0.000,0.333333,0.888889,0.304348,5.274795
19,False,ESMFold2-Experimental-Fast-base300M-step1500k,0,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,5.476138,None,0.554914,NaN,None,SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...,None,GLTPRQVALLVFLTALLALAVSSSGGSSGSGKLAKAVGTAVTYAVT...,False,9,1,0.250,0.500000,0.444444,0.000000,10.289686


In [19]:
df_select

designed_sequence  \
target_name binder_name                                                         
NaN         NaN         27  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        3   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        2   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        5   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        21  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        45  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        25  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        37  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        39  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        40  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        0   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        49  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        23  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        12  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        9   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        19  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        24  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        22  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        14  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        38  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        35  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        4   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        41  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        6   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        47  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        36  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        11  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        17  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        46  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        28  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        32  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        16  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        48  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        10  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        1   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        20  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        13  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        43  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        29  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        8   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        31  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        15  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        33  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        34  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        26  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        18  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        7   SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
                        30  SELEKAMVALIDVFHQYSGREGDKHKLKKSELKELINNELSHFLEE...   
          